# Chapter 7 Companion Notebook: Time Series Analysis and Forecasting

This notebook reproduces every worked numerical example from Chapter 7 of *AI in Finance* (`chapter7.tex`): the correlogram, the Augmented Dickey-Fuller test, the AR(1) fit and forecast (with confidence intervals and p-values), multi-step forecasting, exponential smoothing, cointegration and pairs trading, the GARCH(1,1) variance recursion (reusing Chapter 2's AssetA returns), and a walk-forward validation illustration.

## 1. The interest-rate deviation series and its correlogram

In [1]:
import numpy as np
import pandas as pd

X = np.array([9.0, 8.0, 7.0, 7.0, 5.0, 6.0, 4.0, 4.0])
mean_all = X.mean()
denom = np.sum((X - mean_all) ** 2)

def acf(k, x=X, mean=mean_all, denom=denom):
    return np.sum((x[k:] - mean) * (x[:-k] - mean)) / denom

for k in [1, 2, 3]:
    print(f"ACF({k}) = {acf(k):.4f}")

ACF(1) = 0.4973
ACF(2) = 0.2394
ACF(3) = 0.0346


The roughly geometric decay (about halving at each lag) is the correlogram signature of an AR(1) process, matching Figure 7.1 in the chapter.

## 2. Testing for stationarity: the Augmented Dickey-Fuller test (Section 7.4)

Regress $\Delta X_t$ on $X_{t-1}$ and test $H_0: \gamma=0$ (a unit root) against the nonstandard Dickey-Fuller critical value.

In [2]:
dX = X[1:] - X[:-1]
Xlag_adf = X[:-1]

xbar_adf = Xlag_adf.mean()
gamma_hat = np.sum((Xlag_adf - xbar_adf) * (dX - dX.mean())) / np.sum((Xlag_adf - xbar_adf) ** 2)
alpha_hat = dX.mean() - gamma_hat * xbar_adf

fitted_adf = alpha_hat + gamma_hat * Xlag_adf
resid_adf = dX - fitted_adf
dof_adf = len(dX) - 2
se_gamma = np.sqrt(np.sum(resid_adf ** 2) / dof_adf / np.sum((Xlag_adf - xbar_adf) ** 2))
t_stat = gamma_hat / se_gamma

print(f"gamma_hat = {gamma_hat:.4f}")
print(f"SE(gamma_hat) = {se_gamma:.4f}")
print(f"t-statistic = {t_stat:.4f}")

df_critical_5pct = -2.89
print(f"5% Dickey-Fuller critical value: {df_critical_5pct}")
print(f"Reject unit root? {'Yes' if t_stat < df_critical_5pct else 'No'}")

gamma_hat = -0.2903
SE(gamma_hat) = 0.2589
t-statistic = -1.1215
5% Dickey-Fuller critical value: -2.89
Reject unit root? No


## 3. Fitting the AR(1) model and forecasting (Section 7.6)

In [3]:
Xt = X[1:]
Xlag = X[:-1]
xbar, ybar = Xlag.mean(), Xt.mean()

phi = np.sum((Xlag - xbar) * (Xt - ybar)) / np.sum((Xlag - xbar) ** 2)
c = ybar - phi * xbar

fitted = c + phi * Xlag
resid = Xt - fitted
sse = np.sum(resid ** 2)
sst = np.sum((Xt - ybar) ** 2)
r2 = 1 - sse / sst

print(f"phi = {phi:.4f}, c = {c:.4f}, R^2 = {r2:.4f}")
print(f"Implied long-run mean c/(1-phi) = {c/(1-phi):.3f}")
print(f"Sample mean of full series: {mean_all:.2f}")

X9_forecast = c + phi * X[-1]
print(f"One-step-ahead forecast X9: {X9_forecast:.3f}")

phi = 0.7097, c = 1.1935, R^2 = 0.6005
Implied long-run mean c/(1-phi) = 4.111
Sample mean of full series: 6.25
One-step-ahead forecast X9: 4.032


### Confidence intervals and p-values for $\hat\phi$ (Section 7.6.1)

Also verify $\hat\gamma = \hat\phi - 1$, connecting back to the ADF test above.

In [4]:
from scipy import stats

dof_ar1 = len(Xt) - 2
sigma2_ar1 = sse / dof_ar1
se_phi = np.sqrt(sigma2_ar1 / np.sum((Xlag - xbar) ** 2))
se_c = np.sqrt(sigma2_ar1 * (1 / len(Xt) + xbar ** 2 / np.sum((Xlag - xbar) ** 2)))

t_phi = phi / se_phi
t_c = c / se_c
p_phi = 2 * (1 - stats.t.cdf(abs(t_phi), dof_ar1))
p_c = 2 * (1 - stats.t.cdf(abs(t_c), dof_ar1))
tcrit_ar1 = stats.t.ppf(0.975, dof_ar1)
ci_phi = (phi - tcrit_ar1 * se_phi, phi + tcrit_ar1 * se_phi)

print(f"SE(phi)={se_phi:.4f}, t={t_phi:.2f}, p={p_phi:.4f}")
print(f"95% CI for phi: ({ci_phi[0]:.2f}, {ci_phi[1]:.2f})")
print(f"SE(c)={se_c:.4f}, t={t_c:.2f}, p={p_c:.4f}")
print(f"\nCheck: gamma_hat={gamma_hat:.4f}, phi-1={phi-1:.4f}")

SE(phi)=0.2589, t=2.74, p=0.0407
95% CI for phi: (0.04, 1.38)
SE(c)=1.7503, t=0.68, p=0.5256

Check: gamma_hat=-0.2903, phi-1=-0.2903


## 4. Multi-step forecasting and the growth of uncertainty (Section 7.7)

Recursively forecast horizons $h=1,2,3,5,10$ from $X_8=4.0$, and check convergence toward the long-run mean.

In [5]:
horizons = [1, 2, 3, 5, 10]
forecasts = {}
x_current = X[-1]
h_forecast = 0
x_h = x_current
for h in range(1, max(horizons) + 1):
    x_h = c + phi * x_h
    if h in horizons:
        forecasts[h] = x_h

for h, val in forecasts.items():
    print(f"h={h:2d} -> X_hat_(8+h) = {val:.2f}")

long_run_mean = c / (1 - phi)
print(f"\nLong-run mean c/(1-phi) = {long_run_mean:.3f}")

variance_ratio_limit = 1 / (1 - phi ** 2)
print(f"Limiting variance ratio 1/(1-phi^2) = {variance_ratio_limit:.2f}")

h= 1 -> X_hat_(8+h) = 4.03
h= 2 -> X_hat_(8+h) = 4.06
h= 3 -> X_hat_(8+h) = 4.07
h= 5 -> X_hat_(8+h) = 4.09
h=10 -> X_hat_(8+h) = 4.11

Long-run mean c/(1-phi) = 4.111
Limiting variance ratio 1/(1-phi^2) = 2.01


## 5. Exponential smoothing (Section 7.8)

In [6]:
alpha = 0.4
smoothed = [X[0]]
for t in range(1, len(X)):
    smoothed.append(alpha * X[t] + (1 - alpha) * smoothed[-1])

next_forecast = alpha * X[-1] + (1 - alpha) * smoothed[-1]
smoothed.append(next_forecast)

comparison = pd.DataFrame({
    'Month': list(range(1, 10)),
    'Observed': list(X) + [np.nan],
    'Exp. smoothed forecast': np.round(smoothed, 3),
})
comparison

,Month,Observed,Exp. smoothed forecast
0,1,9.0,9.000
1,2,8.0,8.600
2,3,7.0,7.960
3,4,7.0,7.576
4,5,5.0,6.546
5,6,6.0,6.327
6,7,4.0,5.396
7,8,4.0,4.838
8,9,NaN,4.503


Compare the exponential-smoothing forecast for month 9 to the AR(1) forecast computed above: the two methods use the data differently and need not agree.

## 6. Cointegration and pairs trading (Section 7.10)

Two hypothetical bank stocks. Test whether the spread is stationary via the Engle-Granger two-step method, then compute a $z$-score for the trading rule.

In [7]:
stock_a = np.array([50, 51, 53, 52, 54, 55], dtype=float)
stock_b = np.array([48, 49, 50, 50, 51, 52], dtype=float)
spread = stock_a - stock_b

print(f"Spread: {spread}")
print(f"Mean spread: {spread.mean():.2f}, Std spread: {spread.std(ddof=1):.3f}")

# Engle-Granger step 1: regress A on B
b_bar, a_bar = stock_b.mean(), stock_a.mean()
beta_hat = np.sum((stock_b - b_bar) * (stock_a - a_bar)) / np.sum((stock_b - b_bar) ** 2)
alpha_hat_eg = a_bar - beta_hat * b_bar
print(f"\nCointegrating regression: alpha_hat={alpha_hat_eg:.2f}, beta_hat={beta_hat:.2f}")

# z-score of the final spread
z_final = (spread[-1] - spread.mean()) / spread.std(ddof=1)
print(f"\nFinal spread z-score: {z_final:.2f}")

Spread: [2. 2. 3. 2. 3. 3.]
Mean spread: 2.50, Std spread: 0.548

Cointegrating regression: alpha_hat=-12.50, beta_hat=1.30

Final spread z-score: 0.91


## 7. GARCH(1,1) variance recursion using Chapter 2's AssetA returns (Section 7.12)

In [8]:
r = np.array([0.0200, -0.0098, 0.0297, -0.0096])
omega, alpha_g, beta_g = 0.00002, 0.10, 0.85

sigma2 = np.zeros(5)
sigma2[0] = 0.000414  # seed with the sample variance from Chapter 2
for t in range(1, 5):
    sigma2[t] = omega + alpha_g * r[t - 1] ** 2 + beta_g * sigma2[t - 1]

garch_table = pd.DataFrame({
    't': range(1, 6),
    'sigma^2': sigma2.round(6),
    'sigma (vol)': np.sqrt(sigma2).round(4),
})
print(garch_table.to_string(index=False))

long_run_var = omega / (1 - alpha_g - beta_g)
print(f"\nLong-run variance: {long_run_var:.4f}, long-run volatility: {np.sqrt(long_run_var):.2%}")

 t  sigma^2  sigma (vol)
 1 0.000414       0.0203
 2 0.000412       0.0203
 3 0.000380       0.0195
 4 0.000431       0.0208
 5 0.000396       0.0199

Long-run variance: 0.0004, long-run volatility: 2.00%


Volatility rises from 1.95% to 2.08% immediately after the large return of 2.97% at $t=3$, then decays back toward the long-run level, exactly the volatility-clustering pattern GARCH is designed to capture.

## 8. Walk-forward validation, illustrated (Section 7.14)

Rather than a single train/test split, walk-forward validation refits the model at each step using only data available up to that point, and scores a genuine one-step-ahead forecast each time.

In [9]:
def fit_ar1(x):
    xt, xlag = x[1:], x[:-1]
    xbar_, ybar_ = xlag.mean(), xt.mean()
    phi_ = np.sum((xlag - xbar_) * (xt - ybar_)) / np.sum((xlag - xbar_) ** 2)
    c_ = ybar_ - phi_ * xbar_
    return c_, phi_

errors = []
for cutoff in range(4, len(X)):  # need at least 4 points to fit
    train = X[:cutoff]
    actual_next = X[cutoff]
    c_wf, phi_wf = fit_ar1(train)
    forecast = c_wf + phi_wf * train[-1]
    error = actual_next - forecast
    errors.append(error)
    print(f"Train on months 1-{cutoff}: forecast month {cutoff+1} = {forecast:.2f}, "
          f"actual = {actual_next:.2f}, error = {error:.2f}")

rmse = np.sqrt(np.mean(np.array(errors) ** 2))
print(f"\nWalk-forward RMSE: {rmse:.3f}")

Train on months 1-4: forecast month 5 = 6.83, actual = 5.00, error = -1.83
Train on months 1-5: forecast month 6 = 4.00, actual = 6.00, error = 2.00
Train on months 1-6: forecast month 7 = 6.00, actual = 4.00, error = -2.00
Train on months 1-7: forecast month 8 = 4.07, actual = 4.00, error = -0.07

Walk-forward RMSE: 1.686


Each forecast above uses only data that would genuinely have been available at the time, in contrast to a random $k$-fold split, which could easily train on a later month while validating on an earlier one.

## Exercises (match Chapter 7, Suggested Exercises)

3. Add a ninth observation, $X_9=3.0$, refit the AR(1) model, and compare the new coefficients and forecast to the values above.
4. Compute $\sigma_6^2$ if a fifth AssetA return of $r_5=0.035$ were observed.

In [10]:
# Exercise 3: add a ninth observation and refit
X_ex = np.append(X, 3.0)
c_ex, phi_ex = fit_ar1(X_ex)
forecast_ex = c_ex + phi_ex * X_ex[-1]
print(f"Exercise 3 -- phi={phi_ex:.4f}, c={c_ex:.4f}, forecast X10={forecast_ex:.3f}")

# Exercise 4: extend the GARCH recursion one more step
r5 = 0.035
sigma2_6 = omega + alpha_g * r5 ** 2 + beta_g * sigma2[-1]
print(f"Exercise 4 -- sigma_6^2 = {sigma2_6:.6f}, sigma_6 = {np.sqrt(sigma2_6):.2%}")

Exercise 3 -- phi=0.8085, c=0.4468, forecast X10=2.872
Exercise 4 -- sigma_6^2 = 0.000479, sigma_6 = 2.19%
